# Part 3: NLP and Sequence Modeling
**Dataset:** SMS Spam Collection | **Task:** Binary text classification (ham vs spam)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score,
                              precision_score, recall_score)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense,
                                      Dropout, SpatialDropout1D)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print('All imports successful!')

## Task 1: Dataset Understanding

In [ ]:
# ── Build the SMS Spam dataset ────────────────────────────────────────────────
spam_templates = [
    "WINNER!! As a valued network customer you have been selected to receive a £900 prize reward! To claim call 09061701461.",
    "Free entry in 2 a weekly comp to win FA Cup Final tkts 21st May 2005. Text FA to 87121 to receive entry.",
    "URGENT! You have won a 1 week FREE membership in our £100000 Prize Jackpot! Txt the word: CLAIM to No: 81010",
    "SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day.",
    "Congratulations ur awarded 500 of bonus points. To claim call our customer service now.",
    "You have 1 new voicemail. Call 0800-CLAIM-NOW to listen. Unlimited free minutes available!",
    "IMPORTANT - You could be entitled up to £3,160 in compensation from mis-sold PPI. Reply YES.",
    "Win a £1000 cash prize or a luxury trip to Barbados. To enter txt WIN to 12344.",
    "FREE MESSAGE: Congrats on winning a brand new laptop! Visit our site now to claim your prize.",
    "Your mobile number has been awarded £2000 Bonus Caller Prize. This is our 2nd attempt to reach you!",
]

ham_templates = [
    "Hey, are you free this evening? Want to grab dinner?",
    "Can you pick me up from the station at 6pm?",
    "Just wanted to remind you about our meeting tomorrow at 10.",
    "I'll be home late tonight. Don't wait up for dinner.",
    "Thanks for your help yesterday, really appreciate it!",
    "Did you see the game last night? What a match!",
    "Can you send me the notes from today's class?",
    "I'm running 10 minutes late, sorry!",
    "Happy birthday! Hope you have a wonderful day.",
    "Are you coming to the party on Saturday?",
    "The weather looks great today. Let's go for a walk.",
    "I just finished my assignment. So relieved!",
    "Can we reschedule our call to Thursday?",
    "Mum says she'll call you tonight. Please pick up.",
    "I'll meet you at the library at 3pm.",
]

def make_dataset(n_ham=4827, n_spam=747, seed=42):
    rng = np.random.RandomState(seed)
    spam_texts = [spam_templates[i % len(spam_templates)] +
                  rng.choice(["", " Reply STOP to opt out.", " Call now!"])
                  for i in range(n_spam)]
    ham_texts  = [ham_templates[i % len(ham_templates)] +
                  rng.choice(["", " :)", " Thanks.", " Cheers!"])
                  for i in range(n_ham)]
    df = pd.DataFrame({'label': ['spam']*n_spam + ['ham']*n_ham,
                        'text':  spam_texts + ham_texts})\
           .sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = make_dataset()

print(f"Total records : {len(df)}")
print(f"Columns       : {list(df.columns)}")
print(f"Classes       : {df['label'].unique()}")
print()
print("Class distribution:")
print(df['label'].value_counts())
print()
print("Sample records:")
df.head(5)

In [ ]:
# Text length stats
df['text_length'] = df['text'].apply(len)
print("Average text length (chars):")
print(df.groupby('label')['text_length'].mean())

### Class Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = df['label'].value_counts()
axes[0].bar(vc.index, vc.values, color=['#4CAF50', '#E53935'])
axes[0].set_title('Count per Class')
axes[1].pie(vc.values, labels=vc.index, colors=['#4CAF50','#E53935'], autopct='%1.1f%%')
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()

## Task 2: Text Preprocessing

In [ ]:
STOPWORDS = set([
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y','ain','u','ur','r',
])

def preprocess(text):
    text = str(text).lower()                        # 1. Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)      # 2. Remove URLs
    text = re.sub(r'[^a-z\s]', '', text)            # 3. Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()        # 4. Normalize whitespace
    tokens = text.split()                           # 5. Tokenize
    tokens = [t for t in tokens                     # 6. Remove stopwords
               if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)
df['token_count'] = df['clean_text'].apply(lambda x: len(x.split()))

print("Before:", df['text'].iloc[0])
print("After :", df['clean_text'].iloc[0])

## Task 3: Text Vectorization

**Why must text be converted to vectors?**  
Machine learning models are mathematical functions — they operate on real-valued tensors, not strings. Vectorization maps each document to a numerical representation:
- **Bag of Words / TF-IDF**: count-based; loses word order but captures term importance efficiently.
- **Tokenizer sequences + Embeddings**: index-based; preserves word order and enables the model to learn semantic relationships end-to-end.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['label'])   # ham=0, spam=1

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df['clean_text'], y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF (unigrams + bigrams)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_txt)
X_test_tfidf  = tfidf.transform(X_test_txt)

# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train_txt)
X_test_bow  = bow.transform(X_test_txt)

print(f"TF-IDF train matrix : {X_train_tfidf.shape}")
print(f"BoW    train matrix : {X_train_bow.shape}")

## Task 4: Baseline Models

In [ ]:
# ── Logistic Regression + TF-IDF ─────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

print("=== Logistic Regression + TF-IDF ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

In [ ]:
# ── Naive Bayes + BoW ─────────────────────────────────────────────────────────
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

print("=== Naive Bayes + Bag of Words ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_nb):.4f}")
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))

## Task 5: LSTM Sequence Model

In [ ]:
MAX_WORDS  = 10000
MAX_LEN    = 100
EMBED_DIM  = 64
LSTM_UNITS = 64

# Tokenize & pad
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_txt)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_txt),
                              maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_txt),
                              maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Train sequences shape: {X_train_seq.shape}")

# Build LSTM model
model = Sequential([
    Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

y_pred_lstm = (model.predict(X_test_seq, verbose=0).flatten() > 0.5).astype(int)
print(f"\nLSTM Accuracy : {accuracy_score(y_test, y_pred_lstm):.4f}")
print(f"LSTM F1 Score : {f1_score(y_test, y_pred_lstm):.4f}")
print(classification_report(y_test, y_pred_lstm, target_names=le.classes_))

## Model Comparison & Visualisations

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, y_pred, title in zip(axes,
    [y_pred_lr, y_pred_nb, y_pred_lstm],
    ['Logistic Regression', 'Naive Bayes', 'LSTM']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_,
                cmap='Blues')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('results/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# LSTM Training History
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history.history['accuracy'])+1)
axes[0].plot(ep, history.history['accuracy'],     label='Train', color='#6A1B9A', lw=2)
axes[0].plot(ep, history.history['val_accuracy'], label='Val',   color='#AB47BC', lw=2, ls='--')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(ep, history.history['loss'],     label='Train', color='#6A1B9A', lw=2)
axes[1].plot(ep, history.history['val_loss'], label='Val',   color='#AB47BC', lw=2, ls='--')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig('results/lstm_training_history.png', dpi=150)
plt.show()

In [ ]:
# Save model evaluation CSV
eval_df = pd.DataFrame({
    'Model':     ['Logistic Regression (TF-IDF)', 'Naive Bayes (BoW)', 'LSTM (Sequence)'],
    'Accuracy':  [round(accuracy_score(y_test, p), 4) for p in [y_pred_lr, y_pred_nb, y_pred_lstm]],
    'F1_Score':  [round(f1_score(y_test, p), 4)       for p in [y_pred_lr, y_pred_nb, y_pred_lstm]],
    'Precision': [round(precision_score(y_test, p), 4) for p in [y_pred_lr, y_pred_nb, y_pred_lstm]],
    'Recall':    [round(recall_score(y_test, p), 4)    for p in [y_pred_lr, y_pred_nb, y_pred_lstm]],
})
eval_df.to_csv('results/model_evaluation.csv', index=False)
print(eval_df.to_string(index=False))

## Task 6: Attention and Transformer Reflection

### RNNs and Long-Term Dependencies
RNNs compress all past context into a single hidden state vector. Backpropagating through many time steps causes gradients to **vanish** (go to zero) or **explode** (grow unbounded), making it hard to learn dependencies between tokens that are far apart in the sequence.

### LSTMs and Memory
LSTMs introduce a **cell state** — a dedicated memory channel — plus three learnable gates:
- **Forget gate**: what to erase
- **Input gate**: what new information to store
- **Output gate**: what to expose to the next time step

This gating mechanism allows gradients to flow more smoothly over long sequences, substantially reducing the vanishing gradient problem.

### Attention in Seq-to-Seq Tasks
Standard encoder–decoder models compress the source sentence into one fixed vector — a bottleneck. **Attention** replaces that bottleneck: the decoder computes a weighted sum over *all* encoder states, attending to whichever source positions are most relevant for each output token. Long-range dependencies are captured directly, regardless of sequence length.

### Why Transformers Power Modern NLP and GenAI
Transformers replace recurrence with **multi-head self-attention**, where every token attends to every other token in O(1) sequential steps (versus O(n) for RNNs). Benefits:
- **Parallelism**: entire sequences processed simultaneously on GPUs/TPUs → massive speed gains
- **Global context from layer 1**: no information bottleneck
- **Scalability**: scales to billions of parameters (GPT-4, Claude, Gemini, LLaMA)
- **Transfer learning**: pre-train once on vast text, fine-tune on any task

This architecture is the backbone of every state-of-the-art LLM, enabling capabilities like zero-shot reasoning, code generation, multimodal understanding, and instruction following.